# Microglia Single-Cell RNA-seq Analysis
## Mathys et al., 2017 - Cell Reports

**Paper**: Temporal Tracking of Microglia Activation in Neurodegeneration at Single-Cell Resolution  
**Data**: GEO GSE103334  
**Goal**: Exploratory analysis of microglia cell states during neurodegeneration

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
parent_dir = Path().resolve().parent
sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully!")

## Step 1: Download Data from GEO

We'll use GEOparse to download the data directly from GEO.

In [ ]:
# Install GEOparse if not already installed
# !pip install GEOparse

import GEOparse

# Download GSE103334
print("Downloading data from GEO (this may take a few minutes)...")
gse = GEOparse.get_GEO(geo="GSE103334", destdir="../data/")

print(f"\nGSE Title: {gse.metadata['title'][0]}")
print(f"Number of samples: {len(gse.gsms)}")
print(f"\nSummary: {gse.metadata['summary'][0][:500]}...")

## Step 2: Explore Sample Metadata

In [ ]:
# Extract sample metadata
sample_info = []
for gsm_name, gsm in gse.gsms.items():
    sample_info.append({
        'gsm_id': gsm_name,
        'title': gsm.metadata.get('title', [''])[0],
        'source': gsm.metadata.get('source_name_ch1', [''])[0],
        'treatment': gsm.metadata.get('treatment_protocol_ch1', [''])[0],
    })

metadata_df = pd.DataFrame(sample_info)
print(f"Total samples: {len(metadata_df)}")
metadata_df.head(10)

## Step 3: Download Expression Matrix

The expression data is in supplementary files on the GEO FTP server.

In [ ]:
# List available supplementary files
print("Supplementary files:")
for supp_file in gse.metadata.get('supplementary_file', []):
    print(f"  - {supp_file}")

In [ ]:
# Download supplementary files
import urllib.request
import gzip
import shutil

data_dir = Path("../data/GSE103334")
data_dir.mkdir(parents=True, exist_ok=True)

# Download files from GEO FTP
base_url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE103nnn/GSE103334/suppl/"

# Common file patterns for single-cell data
possible_files = [
    "GSE103334_RAW.tar",
    "GSE103334_count_matrix.txt.gz",
    "GSE103334_expression_matrix.txt.gz",
    "filelist.txt"
]

print("Attempting to download expression data...")
downloaded_files = []

for filename in possible_files:
    url = base_url + filename
    filepath = data_dir / filename
    
    if filepath.exists():
        print(f"✓ {filename} already exists")
        downloaded_files.append(filepath)
        continue
    
    try:
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"✓ Downloaded {filename}")
        downloaded_files.append(filepath)
    except Exception as e:
        print(f"✗ Could not download {filename}: {str(e)[:100]}")

print(f"\nDownloaded {len(downloaded_files)} files")

In [ ]:
# Load expression data
# Look for expression matrix files (common formats: .txt, .csv, .tsv, .txt.gz)

expression_files = list(data_dir.glob("*.txt*")) + list(data_dir.glob("*.csv*")) + list(data_dir.glob("*.tsv*"))

print(f"Found {len(expression_files)} potential expression files:")
for f in expression_files[:10]:
    print(f"  - {f.name}")

# Try to load the first one to see the format
if expression_files:
    test_file = expression_files[0]
    print(f"\nAttempting to load: {test_file.name}")
    
    try:
        if test_file.suffix == '.gz':
            df = pd.read_csv(test_file, compression='gzip', sep='\t', nrows=5)
        else:
            df = pd.read_csv(test_file, sep='\t', nrows=5)
        
        print(f"\nPreview of {test_file.name}:")
        print(f"Shape (first 5 rows): {df.shape}")
        print(df)
    except Exception as e:
        print(f"Error loading file: {e}")
        print("Trying with different separator...")
        try:
            if test_file.suffix == '.gz':
                df = pd.read_csv(test_file, compression='gzip', nrows=5)
            else:
                df = pd.read_csv(test_file, nrows=5)
            print(df)
        except Exception as e2:
            print(f"Still failed: {e2}")
else:
    print("\nNo expression files found. The data might be in a different format.")
    print("Let's check the GEO website directly...")

In [ ]:
# Combined plot - homeostatic vs DAM markers
fig, ax = plt.subplots(figsize=(14, 8))

# Plot homeostatic markers (should decrease)
homeostatic_genes = ['P2ry12', 'Tmem119', 'Cx3cr1']
homeostatic_colors = ['#2E86AB', '#06A77D', '#0B7A75']

# Plot DAM markers (should increase)
dam_genes = ['Apoe', 'Trem2', 'Cd68']
dam_colors = ['#D62828', '#F77F00', '#FCBF49']

time_numeric = [int(tp.replace('w', '')) if 'w' in tp else 0 for tp in time_points]

# Plot homeostatic markers
for gene, color in zip(homeostatic_genes, homeostatic_colors):
    if gene in gene_trajectories:
        data = gene_trajectories[gene]
        ax.plot(time_numeric, data['means'], marker='o', label=f'{gene} (homeostatic)', 
               linewidth=3, markersize=10, color=color, linestyle='-')
        ax.fill_between(time_numeric, 
                       np.array(data['means']) - np.array(data['sems']),
                       np.array(data['means']) + np.array(data['sems']),
                       alpha=0.15, color=color)

# Plot DAM markers  
for gene, color in zip(dam_genes, dam_colors):
    if gene in gene_trajectories:
        data = gene_trajectories[gene]
        ax.plot(time_numeric, data['means'], marker='s', label=f'{gene} (DAM)', 
               linewidth=3, markersize=10, color=color, linestyle='--')
        ax.fill_between(time_numeric, 
                       np.array(data['means']) - np.array(data['sems']),
                       np.array(data['means']) + np.array(data['sems']),
                       alpha=0.15, color=color)

ax.set_xlabel('Time (weeks post-induction)', fontsize=14, fontweight='bold')
ax.set_ylabel('Mean Expression (FPKM)', fontsize=14, fontweight='bold')
ax.set_title('Temporal Evolution: Homeostatic → Disease-Associated Microglia', 
            fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_xticks(time_numeric)
ax.set_xticklabels(time_points, fontsize=12)

# Add annotations
ax.text(0.02, 0.98, 'Homeostatic state', transform=ax.transAxes,
       fontsize=12, verticalalignment='top', 
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
ax.text(0.98, 0.98, 'Reactive state', transform=ax.transAxes,
       fontsize=12, verticalalignment='top', horizontalalignment='right',
       bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

plt.tight_layout()
plt.savefig('../data/homeostatic_vs_dam.png', dpi=300, bbox_inches='tight')
plt.show()

print("\\nKey observations:")
print("- Solid lines (○) = Homeostatic markers (should decrease)")
print("- Dashed lines (□) = DAM markers (should increase)")
print("\\nSaved plot to ../data/homeostatic_vs_dam.png")

In [ ]:
# Plot temporal evolution - one subplot per category
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

categories = list(key_genes.keys())
colors = plt.cm.Set2(range(8))

for idx, category in enumerate(categories):
    ax = axes[idx]
    
    # Plot each gene in this category
    color_idx = 0
    for gene, data in gene_trajectories.items():
        if data['category'] == category:
            # Convert time points to numeric (extract number from '0w', '1w', etc.)
            time_numeric = [int(tp.replace('w', '')) if 'w' in tp else 0 for tp in time_points]
            
            # Plot line with error bars
            ax.plot(time_numeric, data['means'], marker='o', label=gene, 
                   linewidth=2, markersize=8, color=colors[color_idx])
            ax.fill_between(time_numeric, 
                           np.array(data['means']) - np.array(data['sems']),
                           np.array(data['means']) + np.array(data['sems']),
                           alpha=0.2, color=colors[color_idx])
            color_idx += 1
    
    ax.set_xlabel('Time (weeks)', fontsize=12)
    ax.set_ylabel('Mean FPKM', fontsize=12)
    ax.set_title(category, fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(time_numeric)
    ax.set_xticklabels(time_points)

plt.tight_layout()
plt.savefig('../data/temporal_evolution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved plot to ../data/temporal_evolution.png")

In [ ]:
# Select key genes to track over time
key_genes = {
    'Homeostatic markers': ['P2ry12', 'Tmem119', 'Cx3cr1'],
    'DAM markers': ['Apoe', 'Trem2', 'Cd68'],
    'Inflammatory': ['Il1b', 'Tnf', 'Ccl2'],
    'Complement': ['C1qa', 'C1qb', 'C1qc']
}

# Calculate mean expression per time point for each gene
time_points = sorted(cell_meta_df['time_point'].unique())
print(f"Time points in dataset: {time_points}")

# Create data structure for plotting
gene_trajectories = {}

for category, genes in key_genes.items():
    for gene in genes:
        if gene in expr_df.index:
            means = []
            sems = []  # Standard error of the mean
            
            for tp in time_points:
                cells = cell_meta_df[cell_meta_df['time_point'] == tp]['cell_id'].values
                expr_values = expr_df.loc[gene, cells].values
                means.append(np.mean(expr_values))
                sems.append(np.std(expr_values) / np.sqrt(len(expr_values)))
            
            gene_trajectories[gene] = {
                'category': category,
                'means': means,
                'sems': sems
            }

print(f"\nTracking {len(gene_trajectories)} genes across {len(time_points)} time points")

## Step 9: Temporal Evolution of Key Genes

Visualize how important microglia genes change over time during neurodegeneration.

## Step 10: Temporal Analysis by Condition (CK vs CKp25)

Compare gene expression trajectories between control (CK) and disease model (CKp25) conditions.

In [ ]:
# First, check what conditions we have
print("Conditions in dataset:")
condition_counts = cell_meta_df.groupby(['condition', 'time_point']).size().unstack(fill_value=0)
print(condition_counts)

print(f"\nUnique conditions: {sorted(cell_meta_df['condition'].unique())}")
print(f"\nCells per condition:")
print(cell_meta_df['condition'].value_counts())

In [ ]:
# Calculate gene trajectories separately for each condition
conditions = sorted(cell_meta_df['condition'].unique())
time_points = sorted(cell_meta_df['time_point'].unique())

# Use same key genes as before
key_genes = {
    'Homeostatic markers': ['P2ry12', 'Tmem119', 'Cx3cr1'],
    'DAM markers': ['Apoe', 'Trem2', 'Cd68'],
    'Inflammatory': ['Il1b', 'Tnf', 'Ccl2'],
    'Complement': ['C1qa', 'C1qb', 'C1qc']
}

# Create nested structure: condition -> gene -> trajectory
gene_trajectories_by_condition = {}

for condition in conditions:
    gene_trajectories_by_condition[condition] = {}
    
    for category, genes in key_genes.items():
        for gene in genes:
            if gene in expr_df.index:
                means = []
                sems = []
                
                for tp in time_points:
                    # Filter by both condition AND time point
                    mask = (cell_meta_df['condition'] == condition) & (cell_meta_df['time_point'] == tp)
                    cells = cell_meta_df[mask]['cell_id'].values
                    
                    if len(cells) > 0:  # Only calculate if we have cells
                        expr_values = expr_df.loc[gene, cells].values
                        means.append(np.mean(expr_values))
                        sems.append(np.std(expr_values) / np.sqrt(len(expr_values)))
                    else:
                        means.append(np.nan)
                        sems.append(np.nan)
                
                gene_trajectories_by_condition[condition][gene] = {
                    'category': category,
                    'means': means,
                    'sems': sems
                }

print(f"Calculated trajectories for {len(conditions)} conditions across {len(time_points)} time points")
for condition in conditions:
    n_genes = len(gene_trajectories_by_condition[condition])
    print(f"  - {condition}: {n_genes} genes")

In [ ]:
# Comparison plot: Homeostatic vs DAM markers for both conditions
fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

homeostatic_genes = ['P2ry12', 'Tmem119', 'Cx3cr1']
homeostatic_colors = ['#2E86AB', '#06A77D', '#0B7A75']

dam_genes = ['Apoe', 'Trem2', 'Cd68']
dam_colors = ['#D62828', '#F77F00', '#FCBF49']

time_numeric = [int(tp.replace('w', '')) if 'w' in tp else 0 for tp in time_points]

for idx, condition in enumerate(conditions):
    ax = axes[idx]
    
    # Plot homeostatic markers
    for gene, color in zip(homeostatic_genes, homeostatic_colors):
        if gene in gene_trajectories_by_condition[condition]:
            data = gene_trajectories_by_condition[condition][gene]
            # Filter out NaN values
            valid_idx = [i for i, m in enumerate(data['means']) if not np.isnan(m)]
            valid_time = [time_numeric[i] for i in valid_idx]
            valid_means = [data['means'][i] for i in valid_idx]
            valid_sems = [data['sems'][i] for i in valid_idx]
            
            if len(valid_time) > 0:
                ax.plot(valid_time, valid_means, marker='o', label=f'{gene} (homeostatic)', 
                       linewidth=2.5, markersize=9, color=color, linestyle='-', alpha=0.8)
                ax.fill_between(valid_time, 
                               np.array(valid_means) - np.array(valid_sems),
                               np.array(valid_means) + np.array(valid_sems),
                               alpha=0.15, color=color)
    
    # Plot DAM markers  
    for gene, color in zip(dam_genes, dam_colors):
        if gene in gene_trajectories_by_condition[condition]:
            data = gene_trajectories_by_condition[condition][gene]
            # Filter out NaN values
            valid_idx = [i for i, m in enumerate(data['means']) if not np.isnan(m)]
            valid_time = [time_numeric[i] for i in valid_idx]
            valid_means = [data['means'][i] for i in valid_idx]
            valid_sems = [data['sems'][i] for i in valid_idx]
            
            if len(valid_time) > 0:
                ax.plot(valid_time, valid_means, marker='s', label=f'{gene} (DAM)', 
                       linewidth=2.5, markersize=9, color=color, linestyle='--', alpha=0.8)
                ax.fill_between(valid_time, 
                               np.array(valid_means) - np.array(valid_sems),
                               np.array(valid_means) + np.array(valid_sems),
                               alpha=0.15, color=color)
    
    ax.set_xlabel('Time (weeks post-induction)', fontsize=13, fontweight='bold')
    if idx == 0:
        ax.set_ylabel('Mean Expression (FPKM)', fontsize=13, fontweight='bold')
    ax.set_title(f'{condition} Condition', fontsize=15, fontweight='bold')
    ax.legend(fontsize=9, loc='best', framealpha=0.95, ncol=2)
    ax.grid(True, alpha=0.3, linestyle=':')
    ax.set_xticks(time_numeric)
    ax.set_xticklabels(time_points, fontsize=11)

plt.suptitle('Temporal Evolution: Homeostatic vs DAM Markers by Condition', 
            fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/temporal_by_condition.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nKey observations:")
print("- Solid lines (○) = Homeostatic markers")
print("- Dashed lines (□) = DAM markers")
print("- Compare left (CK) vs right (CKp25) to see condition differences")
print("\nSaved plot to ../data/temporal_by_condition.png")

In [ ]:
# Overlay plot: Direct comparison of conditions for key genes
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Select representative genes from each category
representative_genes = {
    'Homeostatic (P2ry12)': 'P2ry12',
    'Homeostatic (Tmem119)': 'Tmem119', 
    'DAM (Apoe)': 'Apoe',
    'DAM (Trem2)': 'Trem2'
}

condition_colors = {
    conditions[0]: '#1f77b4',  # Blue for first condition (likely CK)
    conditions[1]: '#ff7f0e'   # Orange for second condition (likely CKp25)
}

time_numeric = [int(tp.replace('w', '')) if 'w' in tp else 0 for tp in time_points]

for idx, (title, gene) in enumerate(representative_genes.items()):
    ax = axes[idx]
    
    for condition in conditions:
        if gene in gene_trajectories_by_condition[condition]:
            data = gene_trajectories_by_condition[condition][gene]
            
            # Filter out NaN values
            valid_idx = [i for i, m in enumerate(data['means']) if not np.isnan(m)]
            valid_time = [time_numeric[i] for i in valid_idx]
            valid_means = [data['means'][i] for i in valid_idx]
            valid_sems = [data['sems'][i] for i in valid_idx]
            
            if len(valid_time) > 0:
                color = condition_colors[condition]
                ax.plot(valid_time, valid_means, marker='o', label=condition, 
                       linewidth=3, markersize=10, color=color, alpha=0.8)
                ax.fill_between(valid_time, 
                               np.array(valid_means) - np.array(valid_sems),
                               np.array(valid_means) + np.array(valid_sems),
                               alpha=0.2, color=color)
    
    ax.set_xlabel('Time (weeks post-induction)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Expression (FPKM)', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle=':')
    ax.set_xticks(time_numeric)
    ax.set_xticklabels(time_points, fontsize=11)

plt.suptitle('Direct Comparison: CK vs CKp25 for Key Markers', 
            fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../data/condition_overlay_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nThis plot shows direct overlay of both conditions for key genes")
print("Expected pattern:")
print("  - Homeostatic markers: should remain stable in CK, decrease in CKp25")
print("  - DAM markers: should remain low in CK, increase in CKp25")
print("\nSaved plot to ../data/condition_overlay_comparison.png")

## Step 4: Load Expression Matrix

In [ ]:
# Load the expression matrix
data_file = Path("../data/GSE103334/GSE103334_FPKM_CKP25_TOPHAT.txt.gz")

print("Loading expression data...")
expr_df = pd.read_csv(data_file, sep='\t', compression='gzip', index_col=0)

print(f"Expression matrix shape: {expr_df.shape}")
print(f"  - {expr_df.shape[0]:,} genes")
print(f"  - {expr_df.shape[1]:,} cells")
print(f"\nFirst few genes and cells:")
expr_df.iloc[:5, :5]

## Step 5: Parse Sample Information

Extract time points and conditions from cell names.

In [ ]:
# Parse cell metadata from column names
# Format: CK_0w_m3_A1 -> CK (condition), 0w (time point), m3 (mouse), A1 (well)

cell_metadata = []
for cell_name in expr_df.columns:
    parts = cell_name.split('_')
    if len(parts) >= 4:
        cell_metadata.append({
            'cell_id': cell_name,
            'condition': parts[0],
            'time_point': parts[1],
            'mouse': parts[2],
            'well': parts[3]
        })

cell_meta_df = pd.DataFrame(cell_metadata)

print(f"Parsed metadata for {len(cell_meta_df)} cells")
print(f"\nTime points: {sorted(cell_meta_df['time_point'].unique())}")
print(f"Conditions: {cell_meta_df['condition'].unique()}")
print(f"Number of mice: {cell_meta_df['mouse'].nunique()}")

# Count cells per time point
print(f"\nCells per time point:")
print(cell_meta_df['time_point'].value_counts().sort_index())

## Step 6: Basic Quality Control

Check expression levels and filter low-quality cells/genes.

In [ ]:
# Calculate QC metrics
cell_meta_df['total_counts'] = expr_df.sum(axis=0).values
cell_meta_df['n_genes_detected'] = (expr_df > 0).sum(axis=0).values

# Plot QC metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total expression per cell
axes[0].hist(cell_meta_df['total_counts'], bins=50, edgecolor='black')
axes[0].set_xlabel('Total FPKM per cell')
axes[0].set_ylabel('Number of cells')
axes[0].set_title('Distribution of Total Expression')
axes[0].axvline(cell_meta_df['total_counts'].median(), color='red', linestyle='--', label='Median')
axes[0].legend()

# Number of genes detected per cell
axes[1].hist(cell_meta_df['n_genes_detected'], bins=50, edgecolor='black')
axes[1].set_xlabel('Number of genes detected (FPKM > 0)')
axes[1].set_ylabel('Number of cells')
axes[1].set_title('Genes Detected per Cell')
axes[1].axvline(cell_meta_df['n_genes_detected'].median(), color='red', linestyle='--', label='Median')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Median total FPKM per cell: {cell_meta_df['total_counts'].median():.2f}")
print(f"Median genes detected per cell: {cell_meta_df['n_genes_detected'].median():.0f}")

In [ ]:
# Check how many cells per mouse and time point
print("Cells per mouse:")
print(cell_meta_df.groupby(['time_point', 'mouse']).size().unstack(fill_value=0))

print("\n\nTotal cells per mouse (across all time points):")
print(cell_meta_df['mouse'].value_counts().sort_index())

print("\n\nWell positions used (to see if it's 96-well or 384-well):")
well_letters = [cell.split('_')[-1][0] for cell in cell_meta_df['cell_id']]
well_numbers = [int(cell.split('_')[-1][1:]) for cell in cell_meta_df['cell_id']]
print(f"Row letters: {sorted(set(well_letters))}")
print(f"Column numbers range: {min(well_numbers)} to {max(well_numbers)}")

if max(well_numbers) <= 12 and max(well_letters) <= 'H':
    print("\n→ This is a 96-well plate format (8 rows × 12 columns)")
elif max(well_numbers) <= 24 and max(well_letters) <= 'P':
    print("\n→ This is a 384-well plate format (16 rows × 24 columns)")
else:
    print(f"\n→ Custom plate format")

## Step 7: Identify Key Microglia Marker Genes

Let's find the most relevant genes for microglia biology and neurodegeneration.

In [ ]:
# Define key microglia marker genes
microglia_markers = {
    'Homeostatic': ['Cx3cr1', 'Tmem119', 'P2ry12', 'Gpr34', 'Fcrls', 'Siglech'],
    'Pan-microglia': ['Aif1', 'Csf1r', 'Itgam', 'Hexb'],  # Aif1 = Iba1, Itgam = Cd11b
    'Activated/DAM': ['Apoe', 'Trem2', 'Tyrobp', 'Axl', 'Cd68', 'Lpl', 'Cst7'],  # DAM = Disease-Associated Microglia
    'Inflammatory': ['Il1b', 'Tnf', 'Il6', 'Nos2', 'Ccl2', 'Ccl3', 'Ccl4'],
    'Complement': ['C1qa', 'C1qb', 'C1qc'],
    'Phagocytosis': ['Cd68', 'Cd74', 'Clec7a', 'Mrc1'],
}

# Check which markers are present in the data
print("Marker genes present in dataset:\n")
all_genes = set(expr_df.index)
for category, genes in microglia_markers.items():
    present = [g for g in genes if g in all_genes]
    missing = [g for g in genes if g not in all_genes]
    print(f"{category}:")
    print(f"  ✓ Present ({len(present)}): {', '.join(present)}")
    if missing:
        print(f"  ✗ Missing ({len(missing)}): {', '.join(missing)}")
    print()

In [ ]:
# Find most variable genes (likely most biologically relevant)
gene_var = expr_df.var(axis=1)
gene_mean = expr_df.mean(axis=1)

# Filter out very low expression genes
expressed_genes = gene_mean[gene_mean > 1].index
gene_var_filtered = gene_var[expressed_genes]

# Top 20 most variable genes
top_variable = gene_var_filtered.nlargest(20)

print("Top 20 most variable genes (excluding low expression):\n")
for i, (gene, variance) in enumerate(top_variable.items(), 1):
    mean_expr = gene_mean[gene]
    print(f"{i:2d}. {gene:15s} - var: {variance:8.2f}, mean: {mean_expr:6.2f}")

# Check if these overlap with our markers
marker_genes_flat = [g for genes in microglia_markers.values() for g in genes]
overlap = set(top_variable.index) & set(marker_genes_flat)
print(f"\n{len(overlap)} of top 20 variable genes are known microglia markers: {overlap}")

## Step 8: Differential Expression Analysis

Compare gene expression between time points to identify genes that change during neurodegeneration.

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

# First, let's see what time points we have
print("Available time points:")
time_point_counts = cell_meta_df['time_point'].value_counts().sort_index()
print(time_point_counts)

# Let's compare early vs late time points
# You can modify these based on what time points are actually in your data
early_tp = ['0w']  # Baseline
late_tp = ['6w']    # Late disease stage (adjust based on your data)

print(f"\nWill compare: {early_tp} vs {late_tp}")
print("Run this cell first to see time points, then we'll adjust the comparison in the next cell")

In [ ]:
# Perform differential expression analysis
# Modify early_tp and late_tp based on the time points you saw above

early_tp = ['0w']  # Modify based on your data
late_tp = ['6w']   # Modify based on your data

# Get cells for each condition
early_cells = cell_meta_df[cell_meta_df['time_point'].isin(early_tp)]['cell_id'].values
late_cells = cell_meta_df[cell_meta_df['time_point'].isin(late_tp)]['cell_id'].values

print(f"Comparing {len(early_cells)} early cells vs {len(late_cells)} late cells")

# Get expression data for each group
early_expr = expr_df[early_cells]
late_expr = expr_df[late_cells]

# Calculate statistics for each gene
results = []
for gene in expr_df.index:
    early_vals = early_expr.loc[gene].values
    late_vals = late_expr.loc[gene].values
    
    # Calculate mean expression
    early_mean = np.mean(early_vals)
    late_mean = np.mean(late_vals)
    
    # Log2 fold change (add pseudocount to avoid log(0))
    log2fc = np.log2((late_mean + 1) / (early_mean + 1))
    
    # Wilcoxon rank-sum test (non-parametric, good for single-cell)
    if early_mean > 0.1 or late_mean > 0.1:  # Only test expressed genes
        stat, pval = stats.mannwhitneyu(early_vals, late_vals, alternative='two-sided')
    else:
        pval = 1.0
    
    # Percent of cells expressing the gene
    pct_early = np.sum(early_vals > 0) / len(early_vals) * 100
    pct_late = np.sum(late_vals > 0) / len(late_vals) * 100
    
    results.append({
        'gene': gene,
        'early_mean': early_mean,
        'late_mean': late_mean,
        'log2fc': log2fc,
        'pval': pval,
        'pct_early': pct_early,
        'pct_late': pct_late
    })

# Create results dataframe
de_results = pd.DataFrame(results)

# Multiple testing correction (Benjamini-Hochberg FDR)
de_results['padj'] = multipletests(de_results['pval'], method='fdr_bh')[1]

# Sort by significance
de_results = de_results.sort_values('padj')

print(f"\nDifferential expression analysis complete!")
print(f"Tested {len(de_results)} genes")
print(f"Significant genes (padj < 0.05): {(de_results['padj'] < 0.05).sum()}")
print(f"Significant genes (padj < 0.01): {(de_results['padj'] < 0.01).sum()}")

In [ ]:
# Show top upregulated and downregulated genes
sig_genes = de_results[de_results['padj'] < 0.05].copy()

print("=" * 80)
print("TOP 20 UPREGULATED GENES (Late vs Early)")
print("=" * 80)
upregulated = sig_genes[sig_genes['log2fc'] > 0].nlargest(20, 'log2fc')
for i, row in enumerate(upregulated.itertuples(), 1):
    print(f"{i:2d}. {row.gene:15s} | FC: {2**row.log2fc:6.2f}x | "
          f"padj: {row.padj:.2e} | early: {row.early_mean:6.1f} → late: {row.late_mean:6.1f}")

print("\n" + "=" * 80)
print("TOP 20 DOWNREGULATED GENES (Late vs Early)")
print("=" * 80)
downregulated = sig_genes[sig_genes['log2fc'] < 0].nsmallest(20, 'log2fc')
for i, row in enumerate(downregulated.itertuples(), 1):
    print(f"{i:2d}. {row.gene:15s} | FC: {2**row.log2fc:6.2f}x | "
          f"padj: {row.padj:.2e} | early: {row.early_mean:6.1f} → late: {row.late_mean:6.1f}")

# Check how many of our known markers are differentially expressed
marker_genes_flat = [g for genes in microglia_markers.values() for g in genes]
de_markers = sig_genes[sig_genes['gene'].isin(marker_genes_flat)].sort_values('log2fc', ascending=False)
print(f"\n{'=' * 80}")
print(f"KNOWN MICROGLIA MARKERS THAT ARE DIFFERENTIALLY EXPRESSED ({len(de_markers)} genes)")
print("=" * 80)
for row in de_markers.itertuples():
    direction = "↑" if row.log2fc > 0 else "↓"
    print(f"{direction} {row.gene:15s} | FC: {2**row.log2fc:6.2f}x | padj: {row.padj:.2e}")

In [ ]:
# Volcano plot
fig, ax = plt.subplots(figsize=(12, 8))

# Plot all genes
ax.scatter(de_results['log2fc'], -np.log10(de_results['padj']), 
           alpha=0.3, s=10, c='gray', label='Not significant')

# Highlight significant genes
sig_up = de_results[(de_results['padj'] < 0.05) & (de_results['log2fc'] > 1)]
sig_down = de_results[(de_results['padj'] < 0.05) & (de_results['log2fc'] < -1)]

ax.scatter(sig_up['log2fc'], -np.log10(sig_up['padj']), 
           alpha=0.6, s=20, c='red', label=f'Upregulated ({len(sig_up)})')
ax.scatter(sig_down['log2fc'], -np.log10(sig_down['padj']), 
           alpha=0.6, s=20, c='blue', label=f'Downregulated ({len(sig_down)})')

# Add threshold lines
ax.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.5, alpha=0.5)
ax.axvline(1, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
ax.axvline(-1, color='black', linestyle='--', linewidth=0.5, alpha=0.5)

# Label top genes
top_genes = pd.concat([
    sig_up.nlargest(5, 'log2fc'),
    sig_down.nsmallest(5, 'log2fc')
])

for _, row in top_genes.iterrows():
    ax.annotate(row['gene'], 
                xy=(row['log2fc'], -np.log10(row['padj'])),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, alpha=0.8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))

ax.set_xlabel('Log2 Fold Change (Late / Early)', fontsize=12)
ax.set_ylabel('-Log10 Adjusted P-value', fontsize=12)
ax.set_title('Volcano Plot: Differential Expression Analysis', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Red = Upregulated (FC > 2, padj < 0.05)")
print(f"Blue = Downregulated (FC < 0.5, padj < 0.05)")

In [ ]:
# Heatmap of top differentially expressed genes
top_de_genes = pd.concat([
    sig_genes.nlargest(25, 'log2fc'),  # Top 25 upregulated
    sig_genes.nsmallest(25, 'log2fc')  # Top 25 downregulated
])['gene'].values

# Get expression data for these genes
heatmap_data = expr_df.loc[top_de_genes, early_cells.tolist() + late_cells.tolist()]

# Log transform for visualization
heatmap_data_log = np.log2(heatmap_data + 1)

# Create column colors to show early vs late
col_colors = ['blue'] * len(early_cells) + ['red'] * len(late_cells)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(heatmap_data_log, aspect='auto', cmap='viridis', interpolation='nearest')

# Set ticks
ax.set_yticks(range(len(top_de_genes)))
ax.set_yticklabels(top_de_genes, fontsize=8)
ax.set_xlabel('Cells (Blue=Early, Red=Late)', fontsize=12)
ax.set_title('Top 50 Differentially Expressed Genes', fontsize=14, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Log2(FPKM + 1)', fontsize=10)

# Add colored bar at top to show cell groups
for i, color in enumerate(col_colors):
    ax.add_patch(plt.Rectangle((i, -1), 1, 0.5, facecolor=color, edgecolor='none', clip_on=False))

plt.tight_layout()
plt.show()